# 4-3 캡스톤: 리서치 자동화 멀티 에이전트

**원본 교안:** [4-3-research-automation-multi-agent-system.pdf](./4-3-research-automation-multi-agent-system.pdf) · PDF는 그대로 보존합니다.

이 노트북은 슬라이드의 개념 순서를 따라 설명과 확인 실습을 덧붙인 학습본입니다. 로컬 예제는 표준 라이브러리만 사용합니다. 실제 사내 도구·서버 연결은 강의장 환경에서 별도로 확인하세요.


## 학습 목표

- Supervisor·Researcher·Writer의 역할과 승인 경계를 설명한다.
- MCP, RAGaaS, Deep Agents, Langfuse를 흐름에 배치한다.
- 보고서 품질 항목을 점수와 근거로 검토한다.


> **실습 전 확인**  현재 배포본과 정답본의 구성은 다를 수 있습니다. 실습 파일에 MCP 서버와 Skills가 실제 있는지, RAGaaS에는 `RAG_API_KEY`와 `api-key` 헤더가 전달되는지 확인하세요.


## 전체 Workflow · PDF 1–3쪽

[원본 1~3쪽](./4-3-research-automation-multi-agent-system.pdf#page=1)

주제 입력 → Supervisor가 Researcher 실행 결정 → 웹·RAGaaS·MCP 자료 수집 → 사용자 검토·승인 → Writer 초안 → 최종 평가 순서입니다. 어느 단계가 다음 단계의 입력을 만드는지 표시해 두세요.


## Human-in-the-loop · PDF 4쪽

[원본 4쪽](./4-3-research-automation-multi-agent-system.pdf#page=4)

Researcher 결과를 사용자에게 먼저 보여주고 보고서 작성 여부를 확인합니다. 승인하면 Writer로 재개하고, 거절하면 중단합니다. 거절 상태가 코드에서 실제로 Writer 실행을 막는지 확인해야 합니다.


## 자료 연결 · PDF 5쪽

[원본 5쪽](./4-3-research-automation-multi-agent-system.pdf#page=5)

MCP 서버는 `data` 경로의 로컬 문서를 제공하고, RAGaaS는 사내 지식베이스를 API로 검색합니다. Researcher는 웹과 내부 자료를 함께 사용합니다. 사내망 밖에서 RAGaaS를 실행할 수 있는지 별도로 확인해야 합니다.


## 전문 Agent와 관측 · PDF 6–7쪽

[원본 6~7쪽](./4-3-research-automation-multi-agent-system.pdf#page=6)

슬라이드는 Researcher와 Writer를 `create_deep_agent`로 만들고 각 Skill을 분리합니다. `FilesystemBackend`는 Skill 파일 접근, `MemorySaver`는 실행 상태를 맡습니다. Langfuse `CallbackHandler`는 실행 Trace를, Score는 평가 결과를 남깁니다.


## 보고서 품질 평가 · PDF 8–9쪽

[원본 8~9쪽](./4-3-research-automation-multi-agent-system.pdf#page=8)

Relevance(주제 관련성), Faithfulness(근거 충실성), Completeness(구성 완성도), Clarity(명확성)를 각각 0.0~1.0으로 평가합니다. 항목별 근거와 사람이 읽는 코멘트를 남기고 평균 점수를 계산합니다. 평균만으로 약한 항목을 가리지 마세요.


### 실행해 보기: 승인 경계 확인

실제 API를 호출하지 않는 구조 연습입니다. 거절 시 Writer가 실행되지 않습니다.


In [ ]:
def advance(research: str, approved: bool) -> dict:
    if not approved:
        return {"status": "stopped", "report": None}
    return {"status": "written", "report": f"초안: {research}"}

rejected = advance("자료 요약", approved=False)
approved = advance("자료 요약", approved=True)
assert rejected["report"] is None
print(rejected)
print(approved)


### 실행해 보기: 평가표 읽기

평균뿐 아니라 각 항목의 약점을 확인합니다.


In [ ]:
scores = {"relevance": 0.9, "faithfulness": 0.6, "completeness": 0.8, "clarity": 0.9}
assert all(0 <= score <= 1 for score in scores.values())
print("평균:", round(sum(scores.values()) / len(scores), 2))
print("우선 보완:", min(scores, key=scores.get))


## CodeMate와 페어 프로그래밍

아래 문장을 CodeMate에 전달하고, 생성된 파일과 실행 결과를 직접 검토하세요. 사내 확장의 메뉴 이름이나 동작은 현장에서 확인합니다.

> Chapter02/06-capstone의 README.md와 starter.py를 읽고 현재 구현 가능한 단계와 빠진 단계를 표로 정리해 줘. 먼저 승인 전 Writer 호출을 막는 흐름과 거절 테스트를 제안해 줘. 그다음 자료별 출처 기록, 4개 평가 항목, 점수 근거를 검토할 절차를 제시해 줘. 파일 변경은 내가 검토한 뒤 진행해 줘.


## 결과 검토와 되돌아보기

1. 승인하지 않았을 때 Writer가 실행되지 않았음을 무엇으로 증명할까?
2. 웹 검색과 내부 문서가 충돌하면 보고서에 어떻게 표시할까?
3. 평균 점수가 높아도 Faithfulness가 낮으면 어떤 조치를 취할까?
